# 01 - Macro Data Pipeline and Regime Classification

**Purpose:** download monthly growth and inflation data for the US and the Euro area, and classify each month into one of four macro regimes.

**Outputs:**
- `data/macro_data.csv`: monthly growth and inflation, US and Euro area
- `data/regimes.csv`: monthly regime classification for both regions

**Period:** January 1997 – latest month available for all series

## 1. Data sources

| Series | Region | Source | Measure |
|---|---|---|---|
| CPI, all items (`CPIAUCSL`) | US | FRED | Inflation, year-on-year % |
| HICP, overall index (`HICP.M.U2.N.000000.4D0.ANR`) | Euro area | ECB Data Portal | Inflation, year-on-year % |
| Industrial production (`INDPRO`) | US | FRED | Growth, year-on-year % |
| Industrial production excl. construction (`STBS.M.I10.Y.PROD.NS0020.4D0.N.IX`) | Euro area | ECB Data Portal | Growth, year-on-year % |

Headline inflation is used rather than core, because energy shocks (such as 2022) are exactly the regimes the model needs to capture.

Industrial production is used for growth in both regions, so growth is measured the same way in the US and the Euro area. GDP was not used because it is quarterly, published late and heavily revised.

**Missing data:** October 2025 US CPI was never published because of the federal government shutdown. The missing index value is filled by linear interpolation between September and November 2025.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Support Jupyter launched from either the project root or notebooks/.
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if not (project_root / "notebooks" / "01_data_pipeline.ipynb").is_file():
    raise RuntimeError("Run this notebook from the project root or notebooks directory.")
data_dir = project_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL"
cpi = pd.read_csv(url, index_col=0, parse_dates=True)
cpi["CPIAUCSL"] = cpi["CPIAUCSL"].interpolate(limit_area="inside")

cpi["inflation_yoy"] = cpi["CPIAUCSL"].pct_change(12) * 100
cpi.loc["2025-08":"2025-12"]

### Euro area inflation

The ECB provides the annual rate of change directly, so no calculation is needed.

**Note:** in February 2026, Eurostat introduced major methodological changes to the HICP (new ECOICOP 2 classification, base year 2025 = 100). The ECB discontinued the old `ICP` dataset, which ended in December 2025, and replaced it with the new `HICP` dataset. The pipeline uses the new series, which covers December 1996 to the present.

In [ ]:
import requests
import io

url_hicp = "https://data-api.ecb.europa.eu/service/data/HICP/M.U2.N.000000.4D0.ANR?format=csvdata"
response = requests.get(url_hicp, timeout=30)
response.raise_for_status()
hicp_raw = pd.read_csv(io.StringIO(response.text))

hicp = hicp_raw[["TIME_PERIOD", "OBS_VALUE"]].copy()
hicp.columns = ["date", "ea_inflation_yoy"]
hicp["date"] = pd.to_datetime(hicp["date"])
hicp = hicp.set_index("date").sort_index()
hicp.tail(15)

### Euro area growth

The ECB provides an index, so year-on-year growth is calculated in the same way as for the US (`pct_change(12)`). Data is sorted by date first, because the calculation compares each row with the row 12 positions above.

The series uses a **fixed composition** (all 21 current euro area members across the whole history), while the HICP uses a **changing composition** (countries included as they joined). The difference is small and does not affect the results.

In [ ]:
url_ip = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=INDPRO"
ip = pd.read_csv(url_ip, index_col=0, parse_dates=True)

ip["us_growth_yoy"] = ip["INDPRO"].pct_change(12) * 100
ip.tail(15)

In [ ]:
url_ea_ip = "https://data-api.ecb.europa.eu/service/data/STBS/M.I10.Y.PROD.NS0020.4D0.N.IX?format=csvdata"
response = requests.get(url_ea_ip, timeout=30)
response.raise_for_status()
ea_ip_raw = pd.read_csv(io.StringIO(response.text))

ea_ip = ea_ip_raw[["TIME_PERIOD", "OBS_VALUE"]].copy()
ea_ip.columns = ["date", "ea_ip_index"]
ea_ip["date"] = pd.to_datetime(ea_ip["date"])
ea_ip = ea_ip.set_index("date").sort_index()
ea_ip["ea_ip_growth_yoy"] = ea_ip["ea_ip_index"].pct_change(12) * 100
ea_ip.tail(15)

### Euro area growth: Economic Sentiment Indicator

Industrial production is published about six weeks after the month ends and is noisy. The European Commission's Economic Sentiment Indicator (ESI) is published at the end of the month it describes, and covers industry (40%), services (30%), consumers (20%), retail (5%) and construction (5%).

Source: Eurostat, dataset `ei_bssi_m_r2`, series `M.BS-ESI-I.SA.EA21` (euro area, 21 countries, seasonally adjusted). The `EA20` series ended in December 2025, when Bulgaria joined the euro.

The ESI is a survey score with a long-term average of 100, not an output index, so no growth rate is calculated: the 12-month rule is applied to its level directly.

In [ ]:
url_esi = "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/ei_bssi_m_r2/M.BS-ESI-I.SA.?format=SDMX-CSV"
response = requests.get(url_esi, timeout=60)
response.raise_for_status()
esi_raw = pd.read_csv(io.StringIO(response.text))

esi_raw["geo"].unique()

In [ ]:
for area in ["EA20", "EA21"]:
    s = esi_raw.loc[esi_raw["geo"] == area].set_index("TIME_PERIOD")["OBS_VALUE"].dropna()
    print(area, "|", s.index.min(), "to", s.index.max(), "|", len(s), "months")

In [ ]:
esi = esi_raw.loc[esi_raw["geo"] == "EA21", ["TIME_PERIOD", "OBS_VALUE"]].copy()
esi.columns = ["date", "ea_esi"]
esi["date"] = pd.to_datetime(esi["date"])
esi = esi.set_index("date").sort_index()["ea_esi"].dropna()

esi.tail()

In [ ]:
compare = pd.DataFrame({
    "ea_ip_growth": ea_ip["ea_ip_growth_yoy"],
    "ea_esi": esi,
}).loc["1997":].dropna()

for col in compare.columns:
    avg_12m = compare[col].rolling(12).mean()
    rising = (compare[col] > avg_12m).where(avg_12m.notna()).dropna().astype(bool)
    switches = (rising != rising.shift(1)).iloc[1:].sum()
    print(f"{col}: {switches} switches, average regime {len(rising) / (switches + 1):.1f} months")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

compare["ea_ip_growth"].plot(ax=axes[0], title="Euro area industrial production growth (year-on-year, %)")
compare["ea_ip_growth"].rolling(12).mean().plot(ax=axes[0], linestyle="--", label="12-month average")

compare["ea_esi"].plot(ax=axes[1], title="Euro area Economic Sentiment Indicator (long-term average = 100)")
compare["ea_esi"].rolling(12).mean().plot(ax=axes[1], linestyle="--", label="12-month average")

for ax in axes:
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
check = compare.loc["2022-06":"2022-12"].copy()
check["esi_12m_avg"] = compare["ea_esi"].rolling(12).mean()
check

## 2. Combined table

The four series are combined into one monthly table. Only months with data for all four series are kept, so:
- the table **starts** when the shortest series starts (Euro area HICP, 1997)
- the table **ends** at the latest month published for all series

Euro area industrial production is published about six weeks after the month ends, later than the other series. **The model can only know as much as its slowest data.** This publication lag will be applied in the backtest, so the model never uses data before it was actually available.

In [ ]:
data = pd.concat([
    cpi["inflation_yoy"].rename("us_inflation"),
    hicp["ea_inflation_yoy"].rename("ea_inflation"),
    ip["us_growth_yoy"].rename("us_growth"),
    ea_ip["ea_ip_growth_yoy"].rename("ea_growth")
], axis=1)

data=data.dropna()
data.tail(15)

### Plot the combined series

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

data[["us_inflation", "ea_inflation"]].plot(ax=axes[0], title="Inflation (year-on-year, %)")
data[["us_growth", "ea_growth"]].plot(ax=axes[1], title="Industrial production growth (year-on-year, %)")

for ax in axes:
    ax.axhline(0, color="grey", linewidth=0.8)

plt.tight_layout()
plt.show()

In [ ]:
data.to_csv(data_dir / "macro_data.csv")

## 3. Rising or falling?

Each month, growth and inflation are classified as **rising** or **falling**. The model uses the **change** rather than the level: markets already price in the level, so returns respond to whether the economy is accelerating or slowing.

Two rules were tested:
- **12-month rule (main model):** rising if the current value is above its average of the past 12 months
- **3-month rule (comparison):** rising if the current value is above its value 3 months earlier

**Handling missing data:** at the start of the sample, the comparison value does not exist yet. In pandas, comparing a number with a missing value returns `False`, which would silently classify those months as "falling". These months are set to missing and removed instead.

In [ ]:
regimes = pd.DataFrame(index=data.index)

for col in ["us_growth", "us_inflation", "ea_growth", "ea_inflation"]:
    avg_12m = data[col].rolling(12).mean()
    rising = data[col] > avg_12m
    regimes[col + "_rising"] = rising.where(avg_12m.notna())

regimes = regimes.dropna().astype(bool)
regimes.head()

In [ ]:
regimes_3m = pd.DataFrame(index=data.index)

for col in ["us_growth", "us_inflation", "ea_growth", "ea_inflation"]:
    value_3m_ago = data[col].shift(3)
    rising_3m = data[col] > value_3m_ago
    regimes_3m[col + "_rising"] = rising_3m.where(value_3m_ago.notna())

regimes_3m = regimes_3m.dropna().astype(bool)
regimes_3m.head()

## How many times does each method switch

In [ ]:
common_start = regimes.index.min()
r12 = regimes
r3 = regimes_3m.loc[common_start:]

switches_12m = (r12 != r12.shift(1)).iloc[1:].sum()
switches_3m = (r3 != r3.shift(1)).iloc[1:].sum()

comparison = pd.DataFrame({"12-month": switches_12m, "3-month": switches_3m})
comparison

In [ ]:
comparison["12m_avg_length"] = len(r12) / (comparison["12-month"] + 1)
comparison["3m_avg_length"] = len(r3) / (comparison["3-month"] + 1)

comparison.round(1)

### Stability of the two rules

| Variable | Switches (12m) | Switches (3m) | Avg length, months (12m) | Avg length, months (3m) |
|---|---|---|---|---|
| US growth | 43 | 83 | 7.8 | 4.1 |
| US inflation | 51 | 84 | 6.6 | 4.0 |
| EA growth | 55 | 107 | 6.1 | 3.2 |
| EA inflation | 41 | 80 | 8.2 | 4.2 |

The 3-month rule produces regimes lasting only 3–4 months on average, too short to reflect genuine economic phases. The 12-month rule roughly halves the number of switches. **Trade-off:** the 12-month rule filters out noise but recognises turning points later. Both rules will be compared in the backtest.

Euro area growth is the noisiest series under both rules.

## 4. Macro regimes

Growth and inflation are combined into four regimes:

| | Inflation falling | Inflation rising |
|---|---|---|
| **Growth rising** | Goldilocks | Overheating |
| **Growth falling** | Slowdown | Stagflation |

Regimes are built separately for each region, so a missing month in one region does not remove it from the other. The Euro area uses the Economic Sentiment Indicator for growth (see Section 1); the comparison in Section 3 was carried out on industrial production.

In [ ]:
def build_regimes(growth, inflation, prefix):
    df = pd.concat([growth.rename("growth"), inflation.rename("inflation")], axis=1).dropna()

    out = pd.DataFrame(index=df.index)
    for col in ["growth", "inflation"]:
        avg_12m = df[col].rolling(12).mean()
        out[f"{prefix}_{col}_rising"] = (df[col] > avg_12m).where(avg_12m.notna())
    out = out.dropna().astype(bool)

    g = out[f"{prefix}_growth_rising"]
    i = out[f"{prefix}_inflation_rising"]
    conditions = [g & ~i, g & i, ~g & ~i, ~g & i]
    labels = ["Goldilocks", "Overheating", "Slowdown", "Stagflation"]
    out[f"{prefix}_regime"] = np.select(conditions, labels, default="Unknown")
    return out

us_regimes = build_regimes(ip["us_growth_yoy"], cpi["inflation_yoy"], "us")
ea_regimes = build_regimes(esi, hicp["ea_inflation_yoy"], "ea")

regimes = pd.concat([us_regimes, ea_regimes], axis=1).loc["1997":]
regimes.to_csv(data_dir / "regimes.csv")

print("US:", us_regimes.index.min(), "to", us_regimes.index.max())
print("EA:", ea_regimes.index.min(), "to", ea_regimes.index.max())


In [ ]:
regimes.loc["2025-08":"2025-12", ["us_regime", "ea_regime"]]

In [ ]:
regimes.loc["2022-01":"2022-12", ["us_regime", "ea_regime"]]

## 5. Sanity checks

- **2020 (COVID):** both regions classified as **Slowdown** from March to June 2020. (ticked)
- **2022 (energy shock):** both regions classified as **Stagflation** for most of the year. (ticked) The US moved to Slowdown in October 2022, about four months after inflation peaked, showing the lag of the 12-month rule.

## Limitations

- **Lag:** the 12-month rule confirms turning points several months after they happen.
- **Noisy Euro area growth:** Euro area industrial production produced an implausible "Overheating" signal in August–November 2022, during the energy crisis. A broader growth measure (e.g. the European Commission's Economic Sentiment Indicator) could be tested.
- **Revisions:** the data used is the latest revised version, not what was available at the time. Real-time (vintage) data would be more realistic.
- **Industrial production** covers only part of the economy; services are not captured.